# Clase 209 — Prefect 3 + Dagster: el mismo pipeline 2 veces

Mismo pipeline BTC, implementado en Prefect y en Dagster. Comparación lado a lado.

In [ ]:
import os, tempfile, shutil, json
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'flows_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

## 1. Prefect 3 — flow + tasks

In [ ]:
from prefect import flow, task, get_run_logger
from datetime import datetime
import duckdb

DB = str(WORK / 'btc_prefect.duckdb')

@task(retries=3, retry_delay_seconds=5)
def extract() -> dict:
    # stub local: número fake en vez de API
    return {'ts': datetime.utcnow().isoformat(), 'usd': 67500.0}

@task
def load(payload: dict) -> str:
    con = duckdb.connect(DB)
    con.execute('CREATE TABLE IF NOT EXISTS prices (ts TIMESTAMP PRIMARY KEY, usd DOUBLE)')
    con.execute('INSERT OR IGNORE INTO prices VALUES (?, ?)', [payload['ts'], payload['usd']])
    con.close()
    return DB

@task
def transform(db_path: str) -> dict:
    con = duckdb.connect(db_path)
    avg = con.execute('SELECT AVG(usd), COUNT(*) FROM prices').fetchone()
    con.close()
    return {'avg_usd': float(avg[0]) if avg[0] else None, 'n': int(avg[1])}

@flow(name='btc-pipeline')
def btc_pipeline():
    log = get_run_logger()
    metrics = transform(load(extract()))
    log.info(f'metrics: {metrics}')
    return metrics

result = btc_pipeline()
print('resultado:', result)

## 2. Prefect deployment con schedule

In [ ]:
deploy_snippet = '''\
# deploy.py — corré con: python deploy.py
from prefect import serve
from btc_module import btc_pipeline   # importa tu flow

if __name__ == "__main__":
    deployment = btc_pipeline.to_deployment(
        name="btc-hourly",
        cron="0 * * * *",          # cada hora
        tags=["crypto", "prod"],
    )
    serve(deployment)   # mantiene proceso vivo y ejecuta según schedule
'''
print(deploy_snippet)
print('# UI: prefect server start   →   http://localhost:4200')

## 3. Dagster — software-defined assets

In [ ]:
dagster_src = '''\
# dagster_btc.py — corré con: dagster dev -f dagster_btc.py
from dagster import asset, Definitions, AssetExecutionContext, MetadataValue
from datetime import datetime
import duckdb

DB = "/tmp/btc_dagster.duckdb"

@asset
def btc_price() -> dict:
    """Precio actual de BTC scrapeado de la API."""
    return {"ts": datetime.utcnow().isoformat(), "usd": 67500.0}

@asset(deps=[btc_price])
def btc_table(context: AssetExecutionContext, btc_price: dict) -> str:
    """Tabla DuckDB con historial de precios."""
    con = duckdb.connect(DB)
    con.execute("CREATE TABLE IF NOT EXISTS prices (ts TIMESTAMP PRIMARY KEY, usd DOUBLE)")
    con.execute("INSERT OR IGNORE INTO prices VALUES (?, ?)", [btc_price["ts"], btc_price["usd"]])
    n = con.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
    context.add_output_metadata({"rows": MetadataValue.int(n)})
    return DB

@asset(deps=[btc_table])
def daily_avg(context: AssetExecutionContext, btc_table: str) -> float:
    """Promedio diario de precio."""
    con = duckdb.connect(btc_table)
    avg = con.execute("SELECT AVG(usd) FROM prices").fetchone()[0]
    context.add_output_metadata({"avg_usd": MetadataValue.float(float(avg))})
    return float(avg)

defs = Definitions(assets=[btc_price, btc_table, daily_avg])
'''
print(dagster_src)

## 4. Comparativa: mismo pipeline, 3 estilos

In [ ]:
import pandas as pd
comp = pd.DataFrame([
    {'aspect': 'Líneas de código', 'Airflow': '~35 (DAG)', 'Prefect': '~25 (flow)', 'Dagster': '~30 (assets)'},
    {'aspect': 'Modelo mental', 'Airflow': 'task graph', 'Prefect': 'task graph', 'Dagster': 'asset graph'},
    {'aspect': 'Setup local', 'Airflow': 'docker-compose 4 servicios', 'Prefect': 'pip install + 1 cmd', 'Dagster': 'pip install + dagster dev'},
    {'aspect': 'UI / lineage', 'Airflow': 'task-level', 'Prefect': 'task-level + clean', 'Dagster': 'asset-level + freshness'},
    {'aspect': 'Hybrid execution', 'Airflow': 'manual', 'Prefect': 'nativo (workers)', 'Dagster': 'nativo (code locations)'},
    {'aspect': 'Ideal cuando', 'Airflow': 'industria, max madurez', 'Prefect': 'Python-first, equipo chico', 'Dagster': 'data products + dbt'},
])
print(comp.to_string(index=False))

## Ejercicio guiado

1. Levantá `prefect server start` y `python deploy.py`. Confirmá ejecuciones cada minuto (cambiá cron a `*/1 * * * *` para test).
2. Levantá `dagster dev -f dagster_btc.py`. UI en `localhost:3000`. Click "Materialize" sobre `btc_price` → `daily_avg` queda "stale".
3. Materializá `daily_avg`. Confirmá que Dagster re-corre la cadena entera (porque depende de stale).
4. Agregá un check `@asset_check` en Dagster que verifique `avg_usd > 0`. UI muestra el check verde/rojo al lado del asset.
5. Bonus: integrá `dagster-dbt` y agregá un modelo dbt SQL como asset downstream.

## Conclusiones

- Prefect 3 es Airflow refactorizado con Python idiomático moderno; bueno para empezar.
- Dagster cambia el modelo mental: pensás en **datos producidos**, no en **tareas ejecutadas**.
- Para equipos con muchos data products + dbt: Dagster gana.
- Para equipos chicos sin DevOps dedicado: Prefect Cloud free tier es la opción más fácil.